In [1]:
# ==========================
# 1️⃣ Установка зависимостей
# ==========================
!pip install -q transformers accelerate torch pandas
!pip install python-docx
from docx import Document

from transformers import pipeline
import re, json
from google.colab import files

# ==========================
# 2️⃣ Создание pipeline
# ==========================
pipe = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.2", device_map="auto")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Device set to use cuda:0


In [11]:
import re
import json
from docx import Document
from google.colab import files

# ---------- SYSTEM PROMPT ----------
SYSTEM_PROMPT = """Ты — ассистент режиссёра.
Возвращай ТОЛЬКО JSON, начинающийся с { и заканчивающийся }.
Никаких пояснений, комментариев, ```json и текста вне JSON.
Если нет данных — пиши "".
"""

# ---------- USER PROMPT ----------
USER_PROMPT_TEMPLATE = """
Проанализируй следующую сцену и заполни строго этот JSON-шаблон:

{{
  "Серия": "{episode}",
  "НомерСцены": "",
  "Режим": "",
  "Объект": "",
  "Подобъект": "",
  "Синопсис": "",
  "Персонажи": [],
  "Массовка/Группировка": "",
  "Грим/Костюм": "",
  "Реквизит/Игровой транспорт/Животное": "",
  "Декорация": "",
  "Каскадёр/Трюк": "",
  "Администрация/Спецэффект": "",
  "Операторская техника": "",
  "Лед экраны": ""
}}

Текст сцены:
{scene_text}
"""

# ---------- READ DOCX ----------
def read_docx_text(filename):
    doc = Document(filename)
    return "\n".join(p.text for p in doc.paragraphs)

# ---------- SPLIT SCENES ----------
def split_scenes(text):
    pattern = r"(?=(?:^|\n)\d+-\d+(?:-[A-ZА-Я])?\.?\s*(?:НАТ\.|ИНТ\.).*?(?:ДЕНЬ|НОЧЬ|УТРО|ВЕЧЕР))"
    parts = re.split(pattern, text)
    scenes = [p.strip() for p in parts if p.strip()]
    print(scenes)
    return scenes[1:]

# ---------- EXTRACT JSON ----------
def extract_json_from_output(output):
    """Извлекает JSON из ответа Mistral pipeline"""
    if isinstance(output, list):
        text = output[0].get("generated_text", "")
        print(text)
        if isinstance(text, list):  # Mistral chat output format
            for msg in text:
                if msg.get("role") == "assistant":
                    content = msg.get("content", "")
                    break
            else:
                content = ""
        else:
            content = text
    else:
        content = str(output)

    # Удаляем ```json, пробелы и ищем объект
    match = re.search(r"\{.*\}", content, re.DOTALL)
    if not match:
        print("⚠️ JSON не найден в ответе:", content[:300])
        return None
    try:
        return json.loads(match.group(0))
    except Exception as e:
        print("⚠️ Ошибка JSON:", e)
        print("Фрагмент:", match.group(0)[:300])
        return None

# ---------- SCENE ANALYSIS ----------
def extract_from_scene(scene_text, episode, pipe):
    prompt = USER_PROMPT_TEMPLATE.format(scene_text=scene_text, episode=episode)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    output = pipe(messages, max_new_tokens=512)
    return extract_json_from_output(output)

# ---------- PROCESS TEXT ----------
def process_text(text, episode, pipe):
    scenes = split_scenes(text)
    results = []
    for i, scene in enumerate(scenes, start=1):
        print(f"🎬 Сцена {i}/{len(scenes)}")
        data = extract_from_scene(scene, episode, pipe)
        if data:
            results.append(data)
    return results

# ---------- MAIN ----------
print("📁 Загрузите .docx файл сценария")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
text = read_docx_text(filename)

episode_match = re.search(r"(ПЕРВАЯ|ВТОРАЯ|ТРЕТЬЯ)\s+СЕРИЯ", text, re.IGNORECASE)
episode = episode_match.group(1).capitalize() + " серия" if episode_match else "1"

results = process_text(text, episode, pipe)

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ Готово! results.json создан.")

📁 Загрузите .docx файл сценария


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saving ФИШЕР 1 сери __16.05.docx to ФИШЕР 1 сери __16.05 (9).docx
🎬 Сцена 1/1


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.73 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.47 GiB is free. Process 6332 has 13.27 GiB memory in use. Of the allocated memory 13.04 GiB is allocated by PyTorch, and 111.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)